# **Feature Engineering**

In [3]:
from pathlib import Path
import pandas as pd

In [4]:
file_path = Path.cwd().parent / "data" / "raw"
df = pd.read_parquet(file_path / "clean_customer_churn.parquet", engine="fastparquet")
df.head()

,City,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,Los Angeles,Male,No,No,No,2,Yes,No,DSL,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,Los Angeles,Female,No,No,Yes,2,Yes,No,Fiber optic,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,Los Angeles,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,...,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,1,86,5372,Moved
3,Los Angeles,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,Los Angeles,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,1,89,5340,Competitor had better devices


> Let's create bins out of tenure month column

In [5]:
df["tenure_group"] = pd.cut(
    df["Tenure Months"],
    bins=[0, 12, 36, 60, 100],
    labels=["new", "mid", "loyal", "very_loyal"],
)

df.head()

,City,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,...,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason,tenure_group
0,Los Angeles,Male,No,No,No,2,Yes,No,DSL,Yes,...,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer,new
1,Los Angeles,Female,No,No,Yes,2,Yes,No,Fiber optic,No,...,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved,new
2,Los Angeles,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,...,Yes,Electronic check,99.65,820.50,Yes,1,86,5372,Moved,new
3,Los Angeles,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,...,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved,mid
4,Los Angeles,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,...,Yes,Bank transfer (automatic),103.70,5036.30,Yes,1,89,5340,Competitor had better devices,loyal


> Phone Service, Internet service and like this are some sort of services they take so we can convert then into no of services column and drop them.

In [6]:
service_cols = [
    "Phone Service",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
]

df["service_count"] = (df[service_cols] != "No").sum(axis=1)

df.drop(
    [
        "Phone Service",
        "Multiple Lines",
        "Internet Service",
        "Online Security",
        "Online Backup",
        "Device Protection",
        "Tech Support",
        "Streaming TV",
        "Streaming Movies",
    ],
    axis=1,
    inplace=True,
)

df.head()

,City,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason,tenure_group,service_count
0,Los Angeles,Male,No,No,No,2,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer,new,4
1,Los Angeles,Female,No,No,Yes,2,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved,new,2
2,Los Angeles,Female,No,No,Yes,8,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,1,86,5372,Moved,new,6
3,Los Angeles,Female,No,Yes,Yes,28,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved,mid,7
4,Los Angeles,Male,No,No,Yes,49,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,1,89,5340,Competitor had better devices,loyal,7


>One hot encode the object values

In [9]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
# This keeps numerical cols as-is and expands categorical ones
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
df.head()

,Tenure Months,Monthly Charges,Total Charges,Churn Value,Churn Score,CLTV,tenure_group,service_count,City_Acton,City_Adelanto,...,Churn Reason_Limited range of services,Churn Reason_Long distance charges,Churn Reason_Moved,Churn Reason_N/A,Churn Reason_Network reliability,Churn Reason_Poor expertise of online support,Churn Reason_Poor expertise of phone support,Churn Reason_Price too high,Churn Reason_Product dissatisfaction,Churn Reason_Service dissatisfaction
0,2,53.85,108.15,1,86,3239,new,4,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2,70.70,151.65,1,67,2701,new,2,False,False,...,False,False,True,False,False,False,False,False,False,False
2,8,99.65,820.50,1,86,5372,new,6,False,False,...,False,False,True,False,False,False,False,False,False,False
3,28,104.80,3046.05,1,84,5003,mid,7,False,False,...,False,False,True,False,False,False,False,False,False,False
4,49,103.70,5036.30,1,89,5340,loyal,7,False,False,...,False,False,False,False,False,False,False,False,False,False
